# cvfoodid -- End-to-end inference demo

Foto piring -> deteksi bahan -> estimasi massa -> total gizi.

Run this *after* training (see `01_train_colab.ipynb`).

In [ ]:
%cd /content/cv-food-id
!pip install --quiet -e ".[ml,demo]"

## 1. Load the trained detector

In [ ]:
from cvfoodid.detection.detector import IngredientDetector
from cvfoodid.pipeline import FoodPipeline

WEIGHTS = 'runs/detect/cvfoodid-yolo/weights/best.pt'
detector = IngredientDetector(weights_path=WEIGHTS, conf=0.25)
pipeline = FoodPipeline(detector=detector, plate_diameter_mm=240.0)

## 2. Upload a food photo

In [ ]:
from google.colab import files
uploaded = files.upload()
img_path = next(iter(uploaded.keys()))

## 3. Run end-to-end

In [ ]:
out = pipeline.run(img_path)
import json
print(json.dumps(out.as_dict(), indent=2, ensure_ascii=False))

## 4. Visualize

In [ ]:
import cv2, matplotlib.pyplot as plt
img = cv2.imread(img_path)
for d in out.detections:
    x1,y1,x2,y2 = (int(v) for v in d.bbox_xyxy)
    cv2.rectangle(img, (x1,y1), (x2,y2), (0,200,0), 2)
    cv2.putText(img, f'{d.ingredient_id} {d.confidence:.2f}',
                (x1, max(0, y1-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,200,0), 1, cv2.LINE_AA)
plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

## 5. (Optional) Launch the Gradio demo

In [ ]:
!python app/gradio_demo.py --weights runs/detect/cvfoodid-yolo/weights/best.pt --share